In [78]:
from src.utils.data_loader import load_csv

# Group all lag analysis rows by magnitute of rate cut/rate hike
# Small: 0.25 or 25 bps
# Medium: 0.5 or 50 bps
# Large: 0.75-1.0 or 75-100 bps




# We will gather results for all days
# For example we will have small_bucket_spy_day_1, small_bucket_spy_day_3, small_bucket_spy_day_7, small_bucket_spy_day_30
# For each sector, we will have 4 days for each bucket so in total we will have 12 rows for each sector
# In total, we will have 60 rows, because we have 5 sectors, and each sector will have 12 rows. 4 for each bucket
import pandas as pd
import numpy as np

lag_analysis = pd.read_csv("../data/processed/lag_analysis.csv")
lag_analysis['absolute_rate_change'] = lag_analysis['rate_change'].abs()


bins = [0.25, 0.51, 0.75, 1.0]

labels = ['Small', "Medium", "Large"]

lag_analysis['magnitude_bucket'] = pd.cut(lag_analysis['absolute_rate_change'], bins=bins, labels=labels)


lag_analysis.head()




,date,rate_change,rate_hike,rate_cut,spy_day0,spy_day1,spy_day3,spy_day7,spy_day30,xlk_day0,...,xlu_day3,xlu_day7,xlu_day30,xlre_day0,xlre_day1,xlre_day3,xlre_day7,xlre_day30,absolute_rate_change,magnitude_bucket
0,2020-03-04,-0.50,False,True,0.0,-3.32,-12.35,-13.92,-10.27,0.0,...,-7.65,-16.08,-13.81,0.0,-2.29,-10.75,-11.92,-15.91,0.50,Small
1,2020-03-16,-0.85,False,True,0.0,5.40,0.28,3.50,19.83,0.0,...,1.60,0.82,16.87,0.0,6.07,-2.93,-1.80,15.73,0.85,Large
2,2022-03-17,0.25,True,False,0.0,1.10,2.25,3.69,-6.30,0.0,...,-0.05,3.25,0.44,0.0,0.23,-0.22,1.87,-0.54,0.25,NaN
3,2022-05-05,0.50,True,False,0.0,-0.60,-3.56,-3.32,-11.21,0.0,...,-1.17,0.01,-9.45,0.0,-1.02,-7.73,-5.63,-13.71,0.50,Small
4,2022-06-16,0.75,True,False,0.0,0.22,2.55,4.27,12.52,0.0,...,2.02,6.70,13.96,0.0,0.59,4.13,6.62,14.01,0.75,Medium


In [81]:
def categorize_magnitude(rate_change):
    abs_change = abs(rate_change)
    if abs_change <= 0.30:
        return 'Small (0.25bp)'
    elif abs_change <= 0.55:
        return 'Medium (0.50bp)'
    else:
        return 'Large (0.75-1.00bp)'


lag_analysis['magnitude_bucket'] = lag_analysis['rate_change'].apply(categorize_magnitude)


buckets = lag_analysis['magnitude_bucket'].unique()


sectors = ["spy", "xlk", "xlf", "xlu", "xlre"]
days = [1, 3, 7, 30]


lag_analysis.head()

magnitude_effect = pd.DataFrame()


for bucket in buckets:
    bucket_data = lag_analysis[lag_analysis['magnitude_bucket'] == bucket]
    for sector in sectors:
        for day in days:


            col_name = f"{sector}_day{day}"
            total = bucket_data.sum()
            avg_return = bucket_data[col_name].mean()
            std_dev = bucket_data[col_name].std()

            magnitude_effect = pd.concat([magnitude_effect, pd.DataFrame([{
                'magnitude_bucket': bucket,
                'event_count': total,
                'sector': sector,
                'lag_window': f"Day {day}",
                'avg_return': round(avg_return,2),
                'std_dev': round(std_dev,2)
            }])])



key_lags = magnitude_effect[magnitude_effect['lag_window'].isin(['Day 1', 'Day 30'])]



pivot = key_lags.pivot_table(
    index=['sector', 'lag_window'],
    columns='magnitude_bucket',
    values='avg_return',
    aggfunc='first'  # Just take the value (no aggregation needed)
)

pivot.to_csv('../data/processed/magnitude_effect_summary.csv')

